In [ ]:
from databricks.connect import DatabricksSession
import os
import config

os.environ["DATABRICKS_HOST"] = config.DATABRICKS_HOST
os.environ["DATABRICKS_TOKEN"] = config.DATABRICKS_TOKEN
os.environ["DATABRICKS_CLUSTER_ID"] = config.DATABRICKS_CLUSTER_ID
os.environ["DATABRICKS_WORKSPACE_ID"] = config.DATABRICKS_WORKSPACE_ID

spark = DatabricksSession.builder.getOrCreate()

RetriesExceeded: [RETRIES_EXCEEDED] The maximum number of retries has been exceeded.

In [ ]:

from pyspark.ml.feature import StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml import Pipeline

import mlflow
from mlflow.models.signature import infer_signature
from mlflow.exceptions import MlflowException
from pyspark.sql.functions import array, col
from datetime import datetime

In [ ]:
# Catalog name where the model artifacts will be stored in Unity Catalog
CATALOG_NAME = "alexander_booth" 
SCHEMA_NAME = "default" # Schema (database) name within the catalog to organize model artifacts
TABLE_NAME = "breast_cancer_training_data"
MODEL_NAME = "breast_cancer_model"

In [ ]:
# Define configs
input_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"
target_col = "label"
uc_model_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{MODEL_NAME}"

# Base experiment name
today = datetime.now().strftime("%Y%m%d")  # Format: YYYYMMDD
experiment_name = f"breast_cancer_{today}"

In [ ]:
# --- MLflow config ---
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

# Define experiment path (UC)
experiment_path = f"/Users/alexander.booth@databricks.com/{experiment_name}"
artifact_location = "dbfs:/Volumes/alexander_booth/default/mlflow_artifacts"

try:
    mlflow.create_experiment(
        name=experiment_path,
        artifact_location=artifact_location
    )
except MlflowException:
    pass

mlflow.set_experiment(experiment_path)

In [ ]:
# --- Load Data ---
df_raw = spark.read.table(input_table)
df_raw = df_raw.dropna()
label_col = "label"
feature_cols = [c for c in df_raw.columns if c != label_col] # Dynamically select feature columns (exclude 'label')

# Build vectorized dataframe
df_vectorized = df_raw.select(
    col(label_col).cast("double").alias("label"),
    array(*[col(c) for c in feature_cols]).alias("features")
)

train_df, test_df = df_vectorized.randomSplit([0.8, 0.2], seed=42)
train_df.show(5)


In [ ]:
import logging

# Silence the log streaming server warnings
logger = logging.getLogger("pyspark.ml.torch")
logger.setLevel(logging.FATAL)
logger.propagate = False  # prevent bubbling up to root logger

In [ ]:
# --- MLlib via Spark Connect ---
scaler = StandardScaler(inputCol="features", outputCol="scaled_features")
lr = LogisticRegression(numTrainWorkers=2, featuresCol="scaled_features")

pipeline = Pipeline(stages=[scaler, lr])

grid = ParamGridBuilder().addGrid(lr.maxIter, [2, 200]).build()

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=grid,
    parallelism=2,
    evaluator=BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC"),
)

# --- Train + Log ---
now = datetime.now().strftime("%Y%m%d_%H%M%S")  # Format: YYYYMMDD_HHMMSS
run_name = f"logistic-regression-{now}"

with mlflow.start_run(run_name=run_name) as run:
    mlflow.pyspark.ml.autolog()

    model = cv.fit(train_df)
    predictions = model.transform(test_df)

    # Log metrics
    binary_eval = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
    mlflow.log_metric("areaUnderROC", binary_eval.evaluate(predictions))

    for metric in ["accuracy"]:
        evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName=metric)
        mlflow.log_metric(metric, evaluator.evaluate(predictions))

    # Log best model params
    best_model = model.bestModel.stages[1]  # scaler = stage[0], lr = stage[1]
    for param, value in best_model.extractParamMap().items():
        mlflow.log_param(param.name, value)

    # Sample some input + output rows to infer schema
    sample_input = test_df.limit(5).toPandas()
    sample_output = predictions.limit(5).toPandas()[["prediction"]]

    signature = infer_signature(sample_input, sample_output)

    # Explicitly log the model to UC-compatible location
    mlflow.spark.log_model(model.bestModel, artifact_path="model", signature=signature)

    # Register model
    run_id = run.info.run_id
    model_uri = f"runs:/{run_id}/model"

    print(f"Run ID: {run_id}")
    print(f"Model URI: {model_uri}")

    # Create new model version
    mlflow.register_model(
        model_uri=f"runs:/{run_id}/model",
        name="alexander_booth.default.breast_cancer_model"
    )

### ⚠️ Why this fails with Spark Connect

This example uses `pyspark.ml` (the classic Spark MLlib API), which depends on the JVM and requires access to a `SparkContext`. 

Because Spark Connect runs Python code in a client-server model **without an embedded JVM**, attempting to use components like `StandardScaler`, `Pipeline`, or `CrossValidator` from `pyspark.ml` results in the following error:

